# Module P02 — Entités et initialisation

**Objectif** : écrire les fonctions factory `creer_drone()`, `creer_survivant()`, `creer_tempete()` et la fonction `initialiser_partie()`.

Ce notebook est **compatible Google Colab** : aucun import de fichier local.


---
## Section 1 — Rappels synthétiques

### Notions de ce module

| Notion | Définition courte |
|--------|-------------------|
| Fonction factory | Fonction qui crée et retourne un dictionnaire initialisé |
| Valeur par défaut | Valeur fixée par la factory quand le paramètre n'est pas passé |
| `random.choice(liste)` | Tire un élément aléatoire dans une liste |
| `random.randint(a, b)` | Tire un entier aléatoire entre a et b inclus |
| `set()` | Ensemble Python : pas de doublons, accès en O(1) |
| `f"D{i}"` | F-string : concatène une variable dans une chaîne (`D1`, `D2`...) |

### Rappel du module précédent (P01)

En P01, tu as créé `config.py` qui expose `BATTERIE_INIT`, `BATTERIE_MAX`, `GRILLE_TAILLE`...
Ces constantes sont utilisées directement dans les factories de ce module.


In [ ]:
# Constantes simulées (en production, elles viennent de config.py)
import random
BATTERIE_INIT = 10
BATTERIE_MAX  = 20
GRILLE_TAILLE = 12
NB_DRONES     = 3   # réduit pour l'exercice
NB_TEMPETES   = 2
NB_SURVIVANTS = 4
print('Constantes chargées ✅')

---
## Section 2 — Exercice guidé

### Étape 1 — Factory `creer_drone()`

Un drone possède 8 attributs. Complète la fonction factory ci-dessous.


In [ ]:
def creer_drone(identifiant, col, lig):
    """Retourne un dictionnaire représentant un drone."""
    return {
        "id"          : identifiant,
        "col"         : col,
        "lig"         : lig,
        "batterie"    : BATTERIE_INIT,   # valeur initiale depuis config
        "batterie_max": None,  # TODO : complète avec la bonne constante
        "survivant"   : None,  # None = aucun survivant à bord
        "bloque"      : None,  # TODO : valeur initiale (0 ou 1 ?)
        "hors_service": None,  # TODO : valeur initiale (True ou False ?)
    }

In [ ]:
# Vérification creer_drone
d = creer_drone('D1', 3, 5)
assert d['id'] == 'D1', 'id incorrect'
assert d['col'] == 3, 'col incorrect'
assert d['lig'] == 5, 'lig incorrect'
assert d['batterie'] == BATTERIE_INIT, f'batterie initiale doit être {BATTERIE_INIT}'
assert d['batterie_max'] == BATTERIE_MAX, f'batterie_max doit être {BATTERIE_MAX}'
assert d['survivant'] is None, 'survivant doit être None'
assert d['bloque'] == 0, 'bloque doit valoir 0 à la création'
assert d['hors_service'] == False, 'hors_service doit valoir False'
print('✅ creer_drone() validée')

### Étape 2 — Factory `creer_survivant()`

Un survivant possède 4 attributs. Son état initial est toujours `"en_attente"`.


In [ ]:
def creer_survivant(identifiant, col, lig):
    """Retourne un dictionnaire représentant un survivant."""
    # TODO : retourne le dictionnaire avec les 4 attributs (id, col, lig, etat)
    pass

In [ ]:
# Vérification creer_survivant
s = creer_survivant('S1', 7, 2)
assert s['id'] == 'S1'
assert s['col'] == 7 and s['lig'] == 2
assert s['etat'] == 'en_attente', f'etat doit être en_attente, obtenu : {s["etat"]}'
print('✅ creer_survivant() validée')

### Étape 3 — Factory `creer_tempete()`

Une tempête possède 5 attributs. Sa direction `(dx, dy)` est aléatoire,
mais elle ne peut jamais être `(0, 0)` — une tempête doit toujours se déplacer.


In [ ]:
def creer_tempete(identifiant, col, lig):
    """Retourne un dictionnaire représentant une tempête."""
    # TODO : tire dx et dy aléatoirement dans [-1, 0, 1]
    dx = None  # random.choice(...)
    dy = None  # random.choice(...)
    # TODO : garantis que (dx, dy) != (0, 0)
    if dx == 0 and dy == 0:
        pass  # TODO : que faire ici ?    return {"id": identifiant, "col": col, "lig": lig, "dx": dx, "dy": dy}

In [ ]:
# Vérification creer_tempete (100 créations pour tester le cas (0,0))
for _ in range(100):
    t = creer_tempete('T1', 4, 6)
    assert not (t['dx'] == 0 and t['dy'] == 0), 'La tempête ne peut pas avoir (dx=0, dy=0)'
    assert t['dx'] in [-1, 0, 1] and t['dy'] in [-1, 0, 1]
print('✅ creer_tempete() validée (100 tirages aléatoires)')

### Étape 4 — Placement aléatoire sans chevauchement

Avant d'écrire `initialiser_partie()`, compréhends le mécanisme de placement.
Un `set` de positions occupées garantit qu'aucune entité ne se chevauche.


In [ ]:
def _position_aleatoire(occupees, max_tentatives=200):
    """Retourne une position (col, lig) aléatoire non occupée."""
    for _ in range(max_tentatives):
        col = random.randint(0, GRILLE_TAILLE - 1)
        lig = random.randint(0, GRILLE_TAILLE - 1)
        pos = (col, lig)
        if pos not in occupees:
            return pos
    return None  # impossible de placer

# Test du mécanisme
occupees = set()
for i in range(5):
    pos = _position_aleatoire(occupees)
    occupees.add(pos)
    print(f'Position {i+1} : {pos}')
assert len(occupees) == 5, 'Toutes les positions doivent être uniques'
print('✅ Placement sans chevauchement validé')

### Étape 5 — `initialiser_partie()` simplifiée

Assemble les factories et le placement aléatoire pour créer l'état initial.
Cette version simplifiée ne gère pas les bâtiments ni les zones X.


In [ ]:
def initialiser_partie():
    """Crée et retourne un état de jeu simplifié."""
    occupees = set()
    grille = [['.' for _ in range(GRILLE_TAILLE)] for _ in range(GRILLE_TAILLE)]

    # Hôpital
    hopital = _position_aleatoire(occupees)
    occupees.add(hopital)
    grille[hopital[1]][hopital[0]] = 'H'

    # TODO : place NB_DRONES drones avec creer_drone() et _position_aleatoire()
    drones = {}
    # for i in range(1, NB_DRONES + 1):
    #     ...

    # TODO : place NB_TEMPETES tempêtes avec creer_tempete()
    tempetes = {}

    # TODO : place NB_SURVIVANTS survivants avec creer_survivant()
    survivants = {}

    return {
        'tour'        : 1,
        'score'       : 0,
        'partie_finie': False,
        'victoire'    : False,
        'grille'      : grille,
        'hopital'     : hopital,
        'batiments'   : [],
        'drones'      : drones,
        'tempetes'    : tempetes,
        'survivants'  : survivants,
        'zones_x'     : set(),
        'historique'  : [],
    }

In [ ]:
# Vérification de initialiser_partie()
etat = initialiser_partie()

assert etat['tour'] == 1, 'Le tour initial doit être 1'
assert isinstance(etat['grille'], list), 'grille doit être une liste'
assert len(etat['grille']) == GRILLE_TAILLE, f'La grille doit avoir {GRILLE_TAILLE} lignes'
assert len(etat['drones']) == NB_DRONES, f'Attendu {NB_DRONES} drones'
assert len(etat['tempetes']) == NB_TEMPETES, f'Attendu {NB_TEMPETES} tempêtes'
assert len(etat['survivants']) == NB_SURVIVANTS, f'Attendu {NB_SURVIVANTS} survivants'
assert isinstance(etat['zones_x'], set), 'zones_x doit être un set'

# Vérifier que toutes les positions sont uniques
toutes_positions = [etat['hopital']]
toutes_positions += [(d['col'], d['lig']) for d in etat['drones'].values()]
toutes_positions += [(t['col'], t['lig']) for t in etat['tempetes'].values()]
toutes_positions += [(s['col'], s['lig']) for s in etat['survivants'].values()]
assert len(toutes_positions) == len(set(toutes_positions)), 'Des entités se chevauchent !'

print(f'✅ initialiser_partie() validée : {len(toutes_positions)} entités placées sans chevauchement')

---
## ✅ Bilan du module P02

Tu sais maintenant :
- Écrire une fonction factory pour chaque entité du jeu
- Utiliser un `set` pour garantir des positions uniques
- Assembler les factories dans `initialiser_partie()`
- Vérifier l'intégrité de l'état initial avec des `assert`

**Prochaine étape** : Module P03 — Affichage (`affichage.py`)

Pour le corrigé complet avec `initialiser_partie()` complète (bâtiments + zones X), consulte `P02_corrige.py`.
